# Project: The "TechStore" Data Platform

## Part 1: Data Extraction

### Source 1: The ERP System (MySql)

In [1]:
# note for members remeber to install mysql-connector-python using pip install
import mysql.connector
import pandas as pd

#create a connection to the database
db = mysql.connector.connect(
    host="boughida.com",
    user="student_user_4ing",
    password="bi_guelma_2025",
    database="techstore_erp"
)

# create a cursor object to interact with the database, and fetch all table names
mycursor = db.cursor()
mycursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'techstore_erp'")
tables_names = mycursor.fetchall()

# dictionary that will hold dataframes for each table, i used a dictionary instead of direct variable bcz in case when the database is updated (added new table or removed one) the code will still work without any modification
dfs = {}

# iterate over each table name, fetch its data, and store it in a dataframe
for table_name in tables_names:
    mycursor.execute(f"SELECT * FROM {table_name[0]}")
    
    # fetch all rows from the table
    rows = mycursor.fetchall()
    
    # get column names
    cols = [col[0] for col in mycursor.description]
    
    # create a dataframe and store it in the dictionary
    dfs[table_name[0]] = pd.DataFrame(rows, columns=cols)
    
print(dfs.keys())

dict_keys(['table_stores', 'table_customers', 'table_subcategories', 'table_sales', 'table_reviews', 'table_products', 'table_cities', 'table_categories'])


### Source 2: Departmental Files (Excel files)

In [2]:
df_marketing_expenses = pd.read_excel("Excel Files/marketing_expenses.xlsx")
df_monthly_targets = pd.read_excel("Excel Files/monthly_targets.xlsx")
df_shipping_rates = pd.read_excel("Excel Files/shipping_rates.xlsx")

print(df_marketing_expenses.head())
print(df_monthly_targets.head())
print(df_shipping_rates.head())

                  Date     Category Campaign_Type  Marketing_Cost_USD
0  2023-01-01 00:00:00    Computers  Social Media               900.0
1  2023-01-01 00:00:00  Smartphones  Social Media              2123.0
2  2023-01-01 00:00:00        Audio  Social Media               449.0
3  2023-01-01 00:00:00      Cameras            TV               563.0
4  2023-01-01 00:00:00     Printers            TV              1992.0
  Store_ID                Month Target_Revenue    Manager_Name
0       S1  2023-01-01 00:00:00        5429539  Billel Rahmani
1        2  2023-01-01 00:00:00        7808052  Khadidja Talbi
2        3  2023-01-01 00:00:00        4066978    Ryad Guellil
3        4  2023-01-01 00:00:00        3483297      Walid Diaf
4  Store_5  2023-01-01 00:00:00        3478480     Ryad Benali
  region_name    provider  shipping_cost  average_delivery_days
0       South    Yalidine           1005                      7
1        West   Kazi Tour            499                      2
2      Cen

### Source 3: Competitor Pricing (Web Scraping)

In [3]:
# note for members remeber to install requests and beautifulsoup4 using pip install
import requests
from bs4 import BeautifulSoup

# list hold all the website's pages to scrape
pages = ["https://boughida.com/competitor/index.html", "https://boughida.com/competitor/competitor_page_2.html", "https://boughida.com/competitor/competitor_page_3.html"]

rows = []

for URL in pages:
    # retrieve and parse the HTML data for each page
    page = requests.get(URL)
    soup = BeautifulSoup(page.content, "html.parser")
    product_cards = soup.find_all("div", class_="product-card")
    
    for product_card in product_cards:
        # extract product name and price
        product_name = product_card.find(
            "h5", class_="card-title text-primary product-name"
        ).get_text(strip=True)
        product_price = product_card.find(
            "span", class_="product-price"
        ).get_text(strip=True)
        
        # append the extracted data to the rows list
        rows.append({
            "product_name": product_name,
            "product_price": product_price
        })

# store the scraped data in a dataframe
df_product_prices_competitor = pd.DataFrame(rows)
print(df_product_prices_competitor)

                  product_name product_price
0            Samsung S23 Ultra    174800 DZD
1              HP LaserJet Pro     46000 DZD
2   Best Deal: Dell 24 Monitor     24200 DZD
3               Canon i-Sensys     39000 DZD
4                Epson EcoTank     27700 DZD
5           Promo: Nikon D3500     69200 DZD
6         Xiaomi Redmi Note 12     36800 DZD
7          HP Pavilion Desktop     83000 DZD
8                 Redmi Buds 4      5300 DZD
9                   DJI Mini 3    126900 DZD
10        Promo: Sony SRS-XB13     11100 DZD
11              ASUS ROG STRIX    290600 DZD
12                LG UltraGear     41700 DZD
13                IPHONE CABLE      3200 DZD
14             Lenovo ThinkPad     94600 DZD
15                 AirPods Pro     47400 DZD
16          Phone Case Silicon      1800 DZD
17             Sony WH-1000XM5     65400 DZD
18                   iPhone 13    136900 DZD
19             Canon 445 Black      2600 DZD
20                 DJI Mavic 3    350400 DZD
21        

### Source 4: Legacy Archives (OCR)

In [4]:
# note for members remeber to install pytesseract using pip install
from PIL import Image, ImageOps
import pytesseract
import re

images = ["Invoices/order_001.jpg", "Invoices/order_002.jpg", "Invoices/order_003.jpg", "Invoices/order_004.jpg", "Invoices/order_005.jpg"]

rows = []

for img in images:
    # open each image
    image = Image.open(img)
    
    # scale it to grey to improve OCR accuracy (bcz the thinner the text the harder to read it is for OCR)
    gray_image = ImageOps.grayscale(image)
    
    # resizing the dimensions of the image to make text more clear for OCR
    scale_factor = 2
    resized_image = gray_image.resize((gray_image.width * scale_factor, gray_image.height * scale_factor), resample=Image.LANCZOS)
    
    # extract text from the resized image using pytesseract with French language setting
    output = pytesseract.image_to_string(resized_image, lang="fra")

    # using regex library to extract the desired fields from the output
    # date field
    date = re.search(r"Date:\s*([\d-]+)", output)
    date = date.group(1) if date else "Not found"

    # costumerID field
    customer_id = re.search(r"Client ID:\s*(\w+)", output)
    customer_id = customer_id.group(1) if customer_id else "Not found"

    # splitting the output into lines
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    
    # extracting the product details line
    product_line = re.compile(r"^(.+?)\s+(\d+)\s+(\d+)\s+(\d+)$") # regex pattern to match product details line
    for line in lines:
        matched = product_line.match(line) # try to match the current line with the product pattern
        if matched: # if the line matches the pattern, then extract the details
            product_name = matched.group(1).strip()
            quantity = int(matched.group(2))
            total_revenue = int(matched.group(4))
            break
        else:
            # if the current line doesn't match the product details pattern, set default values (to indicate that the extraction failed for this one)
            product_name = quantity = total_revenue = "Not found"

    rows.append({
        "Date": date,
        "CustomerID": customer_id,
        "Product Name": product_name,
        "Quantity": quantity,
        "Total Revenue": total_revenue
    })
    
    
df_invoices = pd.DataFrame(rows)
print(df_invoices)

         Date CustomerID       Product Name  Quantity  Total Revenue
0  2022-09-22      C1001       HP Victus 15         2         250000
1  2022-01-20      C1003     MacBook Air M2         1         195000
2  2022-06-01      C1004  Samsung S23 Ultra         3         555000
3  2022-10-27      C1025      iPhone 14 Pro         3         690000
4  2022-02-27      C1002        Dell XPS 13         2         520000
